In [ ]:

import os
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, '/content/drive/MyDrive/LLM-Groupe-S/Model')

from prepData import PrepareData
from smallGPT import SmallGPT
from trainer import Trainer

# Chemins
MODEL_DIR  = '/content/drive/MyDrive/LLM-Groupe-S/models'
DATA_DIR   = '/content/drive/MyDrive/LLM-Groupe-S/data'
MODEL_PATH = os.path.join(MODEL_DIR, 'best_smallgpt_oz.keras')
TOK_PATH   = os.path.join(MODEL_DIR, 'tokenizer_oz.pkl')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR,  exist_ok=True)

#Prep données
data = PrepareData(sequence_length=20, vocab_size=5000)

text_path = os.path.join(DATA_DIR, 'oz.txt')

if os.path.exists(text_path):
    print("Texte déjà présent")
    text = data.load_from_file(text_path)
else:
    print(" Téléchargement")
    text = data.load_from_url(
        url          = "https://www.gutenberg.org/files/55/55-0.txt",
        start_marker = "Chapter I.",
        end_marker   = "End of Project Gutenberg"
    )
    data.save_text(text, text_path)

train_ds, val_ds = data.build(text)

# Sauvegarder le tokenizer pour pouvoir l'utiliser dans l'intégration du modèle
with open(TOK_PATH, 'wb') as f:
    pickle.dump(data.tokenizer, f)
print(f" Tokenizer sauvegardé : {TOK_PATH}")

#instanciation et compilation du modèle
if os.path.exists(MODEL_PATH):
    print(" Checkpoint trouvé — reprise")
    trainer = Trainer(model=None, model_path=MODEL_PATH)
    model   = trainer.load()
else:
    print("Entraînement from scratch")
    model = SmallGPT(
        sequence_length = 20,
        vocab_size      = data.vocab_size,
        embed_dim       = 64,
        num_heads       = 4,
        ff_dim          = 128,
        num_layers      = 2
    )
    model.compile(
        optimizer = "adam",
        loss      = "sparse_categorical_crossentropy",
        metrics   = ["accuracy"]
    )

#entrainemnt
"""trainer = Trainer(
    model      = model,
    model_path = MODEL_PATH,
    epochs     = 60,
    patience   = 10
)

history = trainer.train(train_ds, val_ds)"""

# 4. Traçage des courbes

def afficher_courbes(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['loss'],     label='train')
    ax1.plot(history.history['val_loss'], label='val')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epochs')
    ax1.legend()

    ax2.plot(history.history['accuracy'],     label='train')
    ax2.plot(history.history['val_accuracy'], label='val')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.legend()

    plt.tight_layout()
    plt.show()

afficher_courbes(history)


# 5. GÉNÉRATION

def generer(model, prompt, tokenizer, sequence_length,
            num_mots=50, temperature=0.8, top_k=10):

    index_to_word = {v: k for k, v in tokenizer.word_index.items()}
    tokens        = tokenizer.texts_to_sequences([prompt])[0]
    generated     = tokens[:]

    for _ in range(num_mots):
        seq        = np.array([generated[-sequence_length:]])
        prediction = model.predict(seq, verbose=0)[0, -1]

        prediction = np.log(prediction + 1e-8) / temperature
        prediction = np.exp(prediction) / np.sum(np.exp(prediction))

        top_k_indices = np.argsort(prediction)[-top_k:]
        top_k_probs   = prediction[top_k_indices]
        top_k_probs  /= np.sum(top_k_probs)

        next_token = np.random.choice(top_k_indices, p=top_k_probs)

        if next_token == tokenizer.word_index.get("<OOV>", 0):
            continue

        generated.append(next_token)

    mots = [index_to_word.get(i, "?") for i in generated]
    return " ".join(mots)

# Tests
print(generer(model, "dorothy looked at", data.tokenizer, 20, num_mots=30))
print(generer(model, "the wizard said", data.tokenizer, 20, num_mots=30))
print(generer(model, "the scarecrow and", data.tokenizer, 20, num_mots=30))

In [2]:
import os
import sys
import pickle
import numpy as np
import tensorflow as tf

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, '/content/drive/MyDrive/LLM-Groupe-S/Model')

from trainer import Trainer
from prepData import PrepareData

# Charger le modèle depuis le drive
MODEL_PATH = '/content/drive/MyDrive/LLM-Groupe-S/models/best_smallgpt_oz.keras'
trainer = Trainer(model=None, model_path=MODEL_PATH)
model = trainer.load()

# Charger le tokenizer
TOK_PATH = '/content/drive/MyDrive/LLM-Groupe-S/models/tokenizer_oz.pkl'
with open(TOK_PATH, 'rb') as f:
    tokenizer = pickle.load(f)

# Recréer data pour avoir vocab_size et sequence_length
data = PrepareData(sequence_length=20, vocab_size=5000)
data.tokenizer = tokenizer
data.index_to_word = {v: k for k, v in tokenizer.word_index.items()}

# 5. GÉNÉRATION

def generer(model, prompt, tokenizer, sequence_length,
            num_mots=50, temperature=0.8, top_k=10):

    index_to_word = {v: k for k, v in tokenizer.word_index.items()}
    tokens        = tokenizer.texts_to_sequences([prompt])[0]
    generated     = tokens[:]

    for _ in range(num_mots):
        seq        = np.array([generated[-sequence_length:]])
        prediction = model.predict(seq, verbose=0)[0, -1]

        prediction = np.log(prediction + 1e-8) / temperature
        prediction = np.exp(prediction) / np.sum(np.exp(prediction))

        top_k_indices = np.argsort(prediction)[-top_k:]
        top_k_probs   = prediction[top_k_indices]
        top_k_probs  /= np.sum(top_k_probs)

        next_token = np.random.choice(top_k_indices, p=top_k_probs)

        if next_token == tokenizer.word_index.get("<OOV>", 0):
            continue

        generated.append(next_token)

    mots = [index_to_word.get(i, "?") for i in generated]
    return " ".join(mots)

# Tests
print(generer(model, "dorothy looked at", data.tokenizer, 20, num_mots=30))
print(generer(model, "the wizard said", data.tokenizer, 20, num_mots=30))
print(generer(model, "the scarecrow and", data.tokenizer, 20, num_mots=30))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Modèle chargé
dorothy looked at her companions and they sat down and looked at the little farther up and flew away with him to the witch’s kiss and dorothy thanked her to go and ask
the wizard said “drink ” “what is it ” asked the lion “well ” said the girl “if we get ready the tin woodman and the lion will find his way to send
the scarecrow and the tin woodman were glad to be of use to these good friends but now that each one of them has had what they had been thinking again and that


In [3]:
model.summary(expand_nested=True)

Model: "small_gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ positional_embedding_1          │ ?                      │       203,328 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_block_2     │ ?                      │        33,280 │
│ (TransformerDecoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_block_3     │ ?                      │        33,280 │
│ (TransformerDecoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_9           │ (None, 20, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 20, 3157)       │       205,205 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,425,665 (5.44 MB)

 Trainable params: 475,221 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 950,444 (3.63 MB)